In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
TARGETVAR  = CONFIGS['domain']['target']
PMAX       = 200.0
SPLITS     = ['train','valid','test']

In [ ]:
statsfile = os.path.join(SPLITSDIR,'stats.json')
with open(statsfile,'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS[f'{TARGETVAR}_mean']
STD  = STATS[f'{TARGETVAR}_std']
ZMIN = (0.0 - MEAN) / STD
ZMAX = (np.log1p(PMAX) - MEAN) / STD
print(f'Target stats: mean={MEAN:.4f}  std={STD:.4f}')
print(f'ZMIN={ZMIN:.4f}  ZMAX={ZMAX:.4f}')

In [ ]:
raw = {}
for split in SPLITS:
    with xr.open_dataset(os.path.join(SPLITSDIR,f'{split}.h5'),engine='h5netcdf') as ds:
        raw[split] = ds[TARGETVAR].load()
    print(f'{split}: {raw[split].sizes}')

In [ ]:
print(f'{"Split":<8} {"Count":>12} {"Min":>10} {"Max":>12} {"Mean":>10} {"Median":>10} {"NaN":>10}')
print('-'*74)
for split in SPLITS:
    vals = raw[split].values.ravel()
    finite = vals[np.isfinite(vals)]
    print(f'{split:<8} {len(finite):>12,} {finite.min():>10.4f} {finite.max():>12.2f} {finite.mean():>10.4f} {np.median(finite):>10.4f} {np.sum(~np.isfinite(vals)):>10,}')

In [ ]:
allvals = np.concatenate([raw[s].values.ravel() for s in SPLITS])
allvals = allvals[np.isfinite(allvals)]
zscored = (np.log1p(allvals) - MEAN) / STD

pcts = [90,95,99,99.5,99.9,99.95,99.99,99.999,100]
print(f'{"Percentile":>12} {"P (mm)":>14} {"z-score":>10} {"Count above":>14}')
print('-'*54)
for p in pcts:
    threshold = np.percentile(allvals,p)
    z = (np.log1p(threshold) - MEAN) / STD
    nabove = np.sum(allvals >= threshold)
    print(f'{p:>12} {threshold:>14.2f} {z:>10.4f} {nabove:>14,}')

In [ ]:
nabove = np.sum(allvals > PMAX)
pctabove = 100.0 * nabove / len(allvals)
print(f'Samples above PMAX ({PMAX} mm): {nabove:,} ({pctabove:.4f}%)')
print(f'Max observed: {allvals.max():.2f} mm')
print(f'\nValues above PMAX:')
above = allvals[allvals > PMAX]
if len(above) > 0:
    print(f'  Count: {len(above):,}')
    print(f'  Min:   {above.min():.2f}')
    print(f'  Max:   {above.max():.2f}')
    print(f'  Mean:  {above.mean():.2f}')
    for thresh in [500,1000,5000,10000,100000]:
        n = np.sum(above > thresh)
        if n > 0:
            print(f'  > {thresh:>7,} mm: {n:,}')

In [ ]:
allda = xr.concat([raw[s] for s in SPLITS],dim='time')
maxmap  = allda.max(dim='time')
nexceed = (allda > PMAX).sum(dim='time')

fig,axs = pplt.subplots(nrows=1,ncols=2,proj='cyl',figwidth=10,share=True)
axs.format(coast=True,latlim=LATRANGE,lonlim=LONRANGE,latlines=[10,15,20],lonlines=[65,75,85],grid=False,
           lonlabels='b',latlabels='l')

m1 = axs[0].pcolormesh(maxmap.lon,maxmap.lat,maxmap,cmap='Reds',vmin=0,levels=12,extend='max')
axs[0].format(title='Maximum observed precipitation')
fig.colorbar(m1,loc='b',col=1,label=f'{TARGETVAR} (mm)')

m2 = axs[1].pcolormesh(nexceed.lon,nexceed.lat,nexceed,cmap='Purples',vmin=0,levels=12,extend='max')
axs[1].format(title=f'Timesteps exceeding {PMAX:.0f} mm')
fig.colorbar(m2,loc='b',col=2,label='Count')

axs.format(abc=True,titleloc='l')
pplt.show()

In [ ]:
nexceed_time = (allda > PMAX).sum(dim=['lat','lon'])

fig,ax = pplt.subplots(figwidth=8,refheight=2)
ax.bar(nexceed_time.time.values,nexceed_time.values,color='#C44E52',alpha=0.8,width=3)
ax.format(grid=False,ylabel=f'Grid cells > {PMAX:.0f} mm',xlabel='Time',
          title='Temporal distribution of extreme values')
pplt.show()

In [ ]:
print('Impact on z-scored training targets:')
print(f'  ZMAX (corresponding to {PMAX} mm): {ZMAX:.4f}')
print(f'  Samples with z > ZMAX: {np.sum(zscored > ZMAX):,}')
print(f'  Max z-score observed:  {zscored.max():.4f}')
print(f'  Max z-score if clipped at PMAX: {ZMAX:.4f}')
print()
print('Loss impact (assuming prediction clamped at ZMAX):')
above_zmax = zscored[zscored > ZMAX]
if len(above_zmax) > 0:
    losses = (ZMAX - above_zmax)**2
    normal_losses = zscored[zscored <= ZMAX]
    print(f'  Mean loss from samples above ZMAX: {losses.mean():.2f}')
    print(f'  Max loss from samples above ZMAX:  {losses.max():.2f}')
    print(f'  These {len(above_zmax):,} samples contribute {losses.sum():.0f} total squared error')
    print(f'  vs {len(normal_losses):,} normal samples')

In [ ]:
clipped = np.clip(allvals,0.0,PMAX)
zclipped = (np.log1p(clipped) - MEAN) / STD

fig,axs = pplt.subplots(nrows=1,ncols=2,figwidth=8,refheight=3)

axs[0].hist(zscored[zscored > ZMIN],bins=200,color='#1B2C61',alpha=0.7,label='Original')
axs[0].hist(zclipped[zclipped > ZMIN],bins=200,color='#F2C75E',alpha=0.5,label=f'Clipped at {PMAX:.0f} mm')
axs[0].axvline(ZMAX,color='#C44E52',linestyle='--',linewidth=1,label=f'ZMAX ({ZMAX:.2f})')
axs[0].format(grid=False,xlabel='z-score',ylabel='Count',title='Full distribution',yscale='log')
axs[0].legend(loc='ur')

axs[1].hist(zscored[zscored > ZMAX],bins=100,color='#1B2C61',alpha=0.7,label='Original')
axs[1].format(grid=False,xlabel='z-score',ylabel='Count',title=f'Tail above ZMAX ({ZMAX:.2f})',yscale='log')
axs[1].legend(loc='ur')

axs.format(abc=True,titleloc='l')
pplt.show()